In [5]:
"""
End-to-end experiment driver for the Neural-Enhanced Contextual
Two-Stage Stochastic Programming food-truck instance.

Pipeline (runs top-to-bottom as a script):

    1. Define the test instance (sizes, costs, demand range, seeds).
    2. Generate historical data and fit the OLS demand model; project it
       onto the (N, I, T) scenario set that SAA / CCG / N-CCG will consume.
    3. Optionally generate MLP training data and train the Q-surrogate.
       Controlled by the boolean `TRAIN_MLP` at the top of the config block:
         - True  : regenerate params.training_data_path AND retrain
                   params.surrogate_path from scratch.
         - False : assume both files already exist and skip this step.
    4. Run SAA, classical CCG, and Neural CCG on the same scenario set
       and print a three-way comparison (objective, iterations, |S_hat|,
       time, relative error vs. SAA).

Run from the Code/ directory so helpers/ / dataset/ / learning/ /
subproblems/ resolve as top-level packages:

    python run_experiment.py
"""
from __future__ import annotations

import time
import numpy as np

from helpers.parameter import Params
from dataset.data_generation import generate_historical_data
from dataset.scenario_generator import generate_scenarios, scenarios_for_method
from learning.ols_fit import fit_demand_model
from learning.nn_training_data import collect_training_data
from learning.mlp_training import train_surrogate
from learning.q_surrogate import QSurrogate

from subproblems.saa import saa
from subproblems.ccg import ccg
from subproblems.neural_ccg import neural_ccg


# =============================================================================
# 0. Configuration
# =============================================================================
# Flip to True to regenerate the MLP training set and retrain the surrogate
# from scratch. Leave at False if the size-tagged artifacts already exist
# from a previous run (paths come from Params; see section 1).
TRAIN_MLP = False

# Independent knobs for the optimization stage.
RUN_SAA    = True
RUN_CCG    = True
RUN_NCCG   = False

# MLP training budget (only used when TRAIN_MLP is True).
MLP_NUM_SAMPLES = 20_000
MLP_HIDDEN      = (128, 128, 64)
MLP_MAX_EPOCHS  = 200

# =============================================================================
# 1. Problem instance
# =============================================================================
print("=" * 78)
print("1. Problem instance")
print("=" * 78)

seed = 27
I, J, T = 10, 20, 3
np.random.seed(seed)
pop_size = np.array(np.random.randint(1, 5, size=I))

acquisition_cost = np.asarray([5, 2, 3])
unmet_penalty    = np.asarray([20, 8, 12])
# revenue          = np.asarray([0,0])
revenue         = np.zeros(T)

params = Params(
    num_areas         = I,
    num_locations     = J,
    num_food_types    = T,
    capacity          = 200.0,
    fixed_cost        = 50.0,
    demand_min        = 0.0,
    demand_max        = 300.0,
    acquisition_cost  = acquisition_cost,
    unmet_penalty     = unmet_penalty,
    revenue           = revenue,
    num_scenarios_saa = 500,        # N used by SAA / CCG / N-CCG below
    time_limit        = 3600.0,
    seed              = seed,
)

print(f"  (I, J, T) = ({I}, {J}, {T})")
print(f"  capacity  = {params.capacity},  fixed_cost = {params.fixed_cost}")
print(f"  N (SAA)   = {params.num_scenarios_saa}")
print(f"  training_data_path = {params.training_data_path}")
print(f"  surrogate_path     = {params.surrogate_path}")

# =============================================================================
# 2. Historical data -> OLS fit -> contextual scenarios
# =============================================================================
print()
print("=" * 78)
print("2. Historical data, OLS fit, and contextual scenarios")
print("=" * 78)

df, truth = generate_historical_data(
    num_areas      = I,
    num_food_types = T,
    num_days       = 90,
    pop_size       = pop_size,
    seed           = seed,
)
print(f"  historical rows: {len(df)}  (num_days=90, I={I}, T={T})")

ols_model = fit_demand_model(df, num_food_types=T, pool_across_food_types=True)
print(f"  OLS fit: pooled across food types; "
      f"residual_std mean = {ols_model.residual_std.mean():.3f}")

# Draw the scenario BUNDLE (contexts + noise + realized d). A single seed
# gives a reproducible set; switching method below changes only the
# projection, not the underlying bundle.
scen = generate_scenarios(
    truth,
    N              = params.num_scenarios_saa,
    num_food_types = T,
    demand_min     = params.demand_min,
    demand_max     = params.demand_max,
    seed           = 1001,
)

# C-2SSP projection: OLS conditional mean + Gaussian residual sampled
# from the OLS-estimated noise scale. Without the residual term we'd get
# only 4 distinct demand vectors (one per (weather, weekday) cell), since
# both context features are binary. The residual restores within-context
# variability, giving N truly distinct scenarios that CCG / N-CCG can
# meaningfully compress.
scenarios = scenarios_for_method(
    scen,
    method        = "contextual_residual",
    model         = ols_model,
    demand_min    = params.demand_min,
    demand_max    = params.demand_max,
    residual_seed = 2027,
)
N = scenarios.shape[0]
unique = len({row.tobytes() for row in scenarios})
print(f"  scenarios projected via method='contextual_residual' (OLS + eps); "
      f"distinct = {unique}/{N}")


# =============================================================================
# 3. MLP: training data collection + training   (TRAIN_MLP gates this block)
# =============================================================================
print()
print("=" * 78)
print("3. Q-surrogate (MLP)")
print("=" * 78)

surrogate = None

if RUN_NCCG:
    if TRAIN_MLP:
        print("  TRAIN_MLP = True -> collecting training data and retraining.")

        print("\n  --- stage 3a: collect training data ------------------------")
        data_summary = collect_training_data(
            params, truth,
            num_samples = MLP_NUM_SAMPLES,
            out_path    = params.training_data_path,
            seed        = seed,
        )
        print(f"  collected {data_summary['num_samples']} samples "
              f"in {data_summary['total_time']:.1f}s "
              f"(Q range [{data_summary['Q_min']:.1f}, {data_summary['Q_max']:.1f}], "
              f"frac negative {data_summary['Q_frac_negative']:.1%}).")

        print("\n  --- stage 3b: train the MLP --------------------------------")
        train_summary = train_surrogate(
            data_path    = params.training_data_path,
            out_path     = params.surrogate_path,
            hidden_sizes = MLP_HIDDEN,
            batch_size   = 256,
            max_epochs   = MLP_MAX_EPOCHS,
            lr           = 1e-3,
            weight_decay = 1e-5,
            patience     = 15,
            seed         = seed,
            verbose      = True,
        )
        print(f"\n  test R^2   = {train_summary['test_r2']:.6f}")
        print(f"  test RMSE  = {train_summary['test_rmse']:.3f}")
        print(f"  surrogate saved to {params.surrogate_path}")
    else:
        print(f"  TRAIN_MLP = False -> skipping data generation and training.")
        print(f"  Expecting an existing surrogate at {params.surrogate_path}.")

    # Load whatever is on disk (the freshly trained one or a previous run).
    surrogate = QSurrogate(params.surrogate_path)
    print(f"  loaded surrogate: input_dim={surrogate.input_dim}, "
          f"hidden={surrogate.hidden_sizes}, device={surrogate.device}")

    if (surrogate.num_areas, surrogate.num_locations, surrogate.num_food_types) \
            != (I, J, T):
        raise SystemExit(
            f"Surrogate was trained on "
            f"(I, J, T)=({surrogate.num_areas}, {surrogate.num_locations}, "
            f"{surrogate.num_food_types}); instance is ({I}, {J}, {T}). "
            "Set TRAIN_MLP=True to retrain for this instance."
        )
else:
    print("  RUN_NCCG = False -> surrogate not loaded.")


# =============================================================================
# 4. Run the three solution methods on the SAME scenario set
# =============================================================================
print()
print("=" * 78)
print("4. Solution methods on the contextual scenario set")
print("=" * 78)

results = {}   # method -> dict(obj, y, w, iters, S_size, time, info)

# ----- 4a. SAA ---------------------------------------------------------------
if RUN_SAA:
    print("\n=== SAA (full MILP; ground-truth optimum for this sample) ===")
    t0 = time.perf_counter()
    obj_saa, y_saa, w_saa, _, _, saa_runtime = saa(scenarios, params, verbose=False)
    elapsed_saa = time.perf_counter() - t0
    print(f"  obj      = {obj_saa:.6f}")
    print(f"  y        = {y_saa.astype(int)}")
    print(f"  runtime  = {saa_runtime:.3f}s  (wall clock {elapsed_saa:.3f}s)")
    results["SAA"] = dict(
        obj     = obj_saa,
        y       = y_saa,
        w       = w_saa,
        iters   = None,
        S_size  = None,
        time    = saa_runtime,
    )

# ----- 4b. Classical CCG -----------------------------------------------------
if RUN_CCG:
    print("\n=== Classical CCG ===")
    obj_ccg, y_ccg, w_ccg, info_ccg = ccg(
        scenarios, params, tolerance=1e-4, print_level=1,
    )
    print(f"  obj       = {obj_ccg:.6f}")
    print(f"  iters     = {info_ccg['iterations']}")
    print(f"  |S_hat|   = {len(info_ccg['S_hat'])}")
    print(f"  LB, UB    = {info_ccg['LB']:.4f}, {info_ccg['UB']:.4f}")
    print(f"  gap       = {info_ccg['gap']:.6e}")
    print(f"  runtime   = {info_ccg['time']:.3f}s")
    print(f"  cvg_flag  = {info_ccg['cvg_flag']}  (1=converged, 2=time, 3=exhausted)")
    results["CCG"] = dict(
        obj     = obj_ccg,
        y       = y_ccg,
        w       = w_ccg,
        iters   = info_ccg["iterations"],
        S_size  = len(info_ccg["S_hat"]),
        time    = info_ccg["time"],
        info    = info_ccg,
    )

# ----- 4c. Neural CCG --------------------------------------------------------
if RUN_NCCG:
    print("\n=== Neural CCG (Shao-style, adapted) ===")
    # compute_true_ub=True gives a true UB and gap AT THE END for diagnostics;
    # it defeats the speedup, so set False in a real benchmark.
    obj_nccg, y_nccg, w_nccg, info_nccg = neural_ccg(
        scenarios, params, surrogate,
        tolerance       = 1e-2,   # relative-violation tolerance (1% of Q spread)
        print_level     = 1,
        compute_true_ub = True,
    )
    print(f"  theta (LB) = {obj_nccg:.6f}")
    print(f"  iters      = {info_nccg['iterations']}")
    print(f"  |S_hat|    = {len(info_nccg['S_hat'])}")
    if "UB_true" in info_nccg:
        print(f"  UB_true    = {info_nccg['UB_true']:.4f}")
        print(f"  gap_true   = {info_nccg['gap_true']:.6e}")
    print(f"  runtime    = {info_nccg['time']:.3f}s")
    print(f"  cvg_flag   = {info_nccg['cvg_flag']}  "
          "(1=NN-term, 2=time, 3=exhausted)")
    results["N-CCG"] = dict(
        obj     = info_nccg.get("UB_true", obj_nccg),
        y       = y_nccg,
        w       = w_nccg,
        iters   = info_nccg["iterations"],
        S_size  = len(info_nccg["S_hat"]),
        time    = info_nccg["time"],
        info    = info_nccg,
    )


# =============================================================================
# 5. Three-way comparison
# =============================================================================
print()
print("=" * 78)
print("5. Summary")
print("=" * 78)

obj_ref = results["SAA"]["obj"] if "SAA" in results else None

header = (f"  {'method':<7}  {'obj':>14}  {'iters':>6}  {'|S_hat|':>8}  "
          f"{'time (s)':>9}  {'rel vs SAA':>11}")
print(header)
print("  " + "-" * (len(header) - 2))

for name in ("SAA", "CCG", "N-CCG"):
    if name not in results:
        continue
    r = results[name]
    obj   = r["obj"]
    iters = "-" if r["iters"]  is None else f"{r['iters']:d}"
    Ssize = "-" if r["S_size"] is None else f"{r['S_size']:d}"
    if obj_ref is not None and obj is not None:
        rel = abs(obj_ref - obj) / max(abs(obj_ref), 1.0)
        rel_str = f"{rel:11.2e}"
    else:
        rel_str = "          -"
    print(
        f"  {name:<7}  {obj:>14.6f}  {iters:>6}  {Ssize:>8}  "
        f"{r['time']:>9.3f}  {rel_str:>11}"
    )

# First-stage agreement with SAA (if available)
if "SAA" in results:
    y_ref = results["SAA"]["y"].astype(int)
    print()
    for name in ("CCG", "N-CCG"):
        if name not in results:
            continue
        y_this = results[name]["y"].astype(int)
        match = np.array_equal(y_ref, y_this)
        print(f"  y_{name:<6} == y_SAA : {match}")
        if not match:
            print(f"    y_SAA   = {y_ref}")
            print(f"    y_{name:<5} = {y_this}")


1. Problem instance
  (I, J, T) = (10, 20, 3)
  capacity  = 200.0,  fixed_cost = 50.0
  N (SAA)   = 500
  training_data_path = data/recourse_training_10I_20J_3T.npz
  surrogate_path     = data/q_surrogate_10I_20J_3T.pt

2. Historical data, OLS fit, and contextual scenarios
  historical rows: 2700  (num_days=90, I=10, T=3)
  OLS fit: pooled across food types; residual_std mean = 13.568
  scenarios projected via method='contextual_residual' (OLS + eps); distinct = 500/500

3. Q-surrogate (MLP)
  RUN_NCCG = False -> surrogate not loaded.

4. Solution methods on the contextual scenario set

=== SAA (full MILP; ground-truth optimum for this sample) ===
  obj      = 26352.565478
  y        = [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  runtime  = 1.590s  (wall clock 5.293s)

=== Classical CCG ===
iter     1 | LB =      0.000 | UB =  59822.642 | gap =   100.00% | |S_hat| =     1 | iter_time =  0.10 s
Not converged at iter 1 with gap 1
iter     2 | LB =    131.962 | UB =  59822.642 | gap =    9